Buat EDA

In [2]:
import pandas as pd
from pathlib import Path

output modelnya bilang teddy istri pak presiden jir, mau di cek apa di dataset ada yang ngomong gitu

In [2]:
import json

input_file = "datasetSFT.jsonl"
output_file = "hasil_teddy.jsonl"
search_keyword = "teddy"

count = 0

# print(9500; Memulai pencarian kata '{search_keyword}'...")

# Buka file input untuk dibaca dan file output untuk ditulis
with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:
    
    for line in infile:
        if not line.strip():
            continue  # Lewati baris kosong jika ada
            
        # Cari kata "teddy" secara langsung di teks baris mentah (case-insensitive)
        if search_keyword.lower() in line.lower():
            outfile.write(line)
            count += 1

print(f"---")
print(f"&#10004; Selesai! Berhasil menemukan {count} data.")
print(f"&#128190; Hasilnya sudah disimpan ke: {output_file}")

---
&#10004; Selesai! Berhasil menemukan 87 data.
&#128190; Hasilnya sudah disimpan ke: hasil_teddy.jsonl


In [22]:
data_gabungan = pd.read_csv("csv_gabungan.csv")

In [23]:
data_gabungan["label"].unique()

array(['Fitnah', 'Disinformasi', 'Fakta', 'Bukan DFK', 'Ujaran Kebencian',
       'Netral', 'netral', 'ujaran_kebencian'], dtype=object)

secara label udah bener

coba kita cek pas udah dibentuk jadi format alpaca

In [24]:
data_SFT = pd.read_json("datasetSFT.jsonl", lines=True)

In [25]:
data_SFT.head()

,instruction,input,output
0,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Guru Besar UGM tuduh kepala daerah ...,Penjelasan: Pernyataan Prof. Wahyudi dipelinti...
1,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'RT ini nih yg mau d panggil yg terh...,Penjelasan: Komentar merupakan ekspresi ketida...
2,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Camat Kepanjenkidul Kota Blitar Ind...,Penjelasan: Ini adalah modus pencatutan nama y...
3,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Gubernur Sulsel cuma mau cari popul...,Penjelasan: Claim tersebut menggunakan framing...
4,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Judul: .\n\nTeks: Warga Madura dike...,Penjelasan:\n* Claim menggeneralisir warga Mad...


In [26]:
data_SFT['label'] = data_SFT['output'].str.extract(r'Label:\s*(.*)', expand=False).str.strip()
data_SFT['label'] = data_SFT['label'].str.rstrip('.')


In [27]:
data_SFT['label'].unique()

array(['Fitnah', 'Netral', 'Disinformasi', 'Ujaran Kebencian'],
      dtype=object)

udah bener juga, sekalian cek dalam bentuk TRL

In [28]:
data_SFT_TRL = pd.read_json("datasetSFT_TRL.jsonl", lines=True)

In [29]:
data_SFT_TRL.head()

,messages
0,"[{'role': 'system', 'content': 'Anda adalah si..."
1,"[{'role': 'system', 'content': 'Anda adalah si..."
2,"[{'role': 'system', 'content': 'Anda adalah si..."
3,"[{'role': 'system', 'content': 'Anda adalah si..."
4,"[{'role': 'system', 'content': 'Anda adalah si..."


In [30]:
import pandas as pd
import json

data_list = []
with open("datasetSFT_TRL.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data_json = json.loads(line)
        
        assistant_content = ""
        for msg in data_json["messages"]:
            if msg["role"] == "assistant":
                assistant_content = msg["content"]
                break
        
        data_list.append({"assistant_response": assistant_content})

df_trl = pd.DataFrame(data_list)

# Ekstrak Label menggunakan Regex dari teks assistant
# Mencari kata 'Label:' lalu mengambil kata setelahnya, dan menghapus titik (.) di akhir
df_trl['label'] = df_trl['assistant_response'].str.extract(r'Label:\s*(.*)', expand=False).str.strip()
df_trl['label'] = df_trl['label'].str.rstrip('.')

# Cetak hasil untuk memastikan
print("Hasil Ekstraksi Label:")
print(df_trl['label'].value_counts()) # Melihat distribusi label yang berhasil diambil
print("\nContoh beberapa baris teratas:")
print(df_trl[['label']].head())

Hasil Ekstraksi Label:
label
Netral              7500
Fitnah              6341
Disinformasi        6341
Ujaran Kebencian    6341
Name: count, dtype: int64

Contoh beberapa baris teratas:
              label
0            Fitnah
1            Netral
2      Disinformasi
3            Fitnah
4  Ujaran Kebencian
